Bước 3: Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import ast
import os

In [2]:
movies = pd.read_csv('../data/processed/movies_clean.csv')
ratings = pd.read_csv('../data/processed/ratings_clean.csv')
links_small = pd.read_csv('../data/raw/links_small.csv')

# Parse lại các cột list
for col in ['genres_clean', 'keywords_clean', 'cast_clean']:
    movies[col] = movies[col].apply(ast.literal_eval)

# Sửa lỗi: chuỗi rỗng bị đọc thành NaN khi load lại CSV
movies['director_clean'] = movies['director_clean'].fillna('')
movies['overview'] = movies['overview'].fillna('')

print(movies.shape)
print("director_clean rỗng:", (movies['director_clean'] == '').sum())

(45429, 21)
director_clean rỗng: 887


Bước 3.1: Tạo "metadata soup" cho Content-Based

In [3]:
def create_soup(row):
    parts = (
        row['genres_clean'] +
        row['keywords_clean'] +
        row['cast_clean'] +
        [row['director_clean']] * 2 +
        str(row['overview']).lower().split()
    )
    return ' '.join(parts)

movies['soup'] = movies.apply(create_soup, axis=1)
movies[['title', 'soup']].head(3)

,title,soup
0,Toy Story,animation comedy family jealousy toy boy frien...
1,Jumanji,adventure fantasy family boardgame disappearan...
2,Grumpier Old Men,romance comedy fishing bestfriend duringcredit...


In [4]:
# Kiểm tra độ dài soup trung bình, có dòng nào rỗng không
movies['soup_length'] = movies['soup'].str.split().str.len()
print(movies['soup_length'].describe())
print("Số phim có soup rỗng:", (movies['soup_length'] == 0).sum())

count    45429.000000
mean        64.125096
std         35.756547
min          0.000000
25%         36.000000
50%         58.000000
75%         84.000000
max        326.000000
Name: soup_length, dtype: float64
Số phim có soup rỗng: 25


Bước 3.2: Vector hóa văn bản

In [5]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

count_vec = CountVectorizer(max_features=5000, stop_words='english')
count_matrix = count_vec.fit_transform(movies['soup'])
print("CountVectorizer matrix:", count_matrix.shape)

tfidf_vec = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_matrix = tfidf_vec.fit_transform(movies['soup'])
print("TF-IDF matrix:", tfidf_matrix.shape)

CountVectorizer matrix: (45429, 5000)
TF-IDF matrix: (45429, 5000)


Bước 3.3: Weighted Rating

In [6]:
C = movies['vote_average'].mean()
m = movies['vote_count'].quantile(0.75)
print(f"C (mean vote_average): {C:.3f}, m (ngưỡng vote_count 75%): {m}")

def weighted_rating(row, m=m, C=C):
    v = row['vote_count']
    R = row['vote_average']
    return (v / (v + m) * R) + (m / (v + m) * C)

movies['weighted_rating'] = movies.apply(weighted_rating, axis=1)
movies[['title', 'vote_average', 'vote_count', 'weighted_rating']].sort_values('weighted_rating', ascending=False).head(10)

C (mean vote_average): 5.618, m (ngưỡng vote_count 75%): 34.0


,title,vote_average,vote_count,weighted_rating
10306,Dilwale Dulhania Le Jayenge,9.1,661.0,8.929680
314,The Shawshank Redemption,8.5,8358.0,8.488325
834,The Godfather,8.5,6024.0,8.483828
40219,Your Name.,8.5,1030.0,8.407920
12477,The Dark Knight,8.3,12269.0,8.292589
2842,Fight Club,8.3,9678.0,8.290612
292,Pulp Fiction,8.3,8670.0,8.289525
39054,Planet Earth,8.8,176.0,8.284892
522,Schindler's List,8.3,4436.0,8.279603
23655,Whiplash,8.3,4376.0,8.279326


Bước 3.4: Ma trận User-Item cho Collaborative Filtering

In [7]:
ratings_mapped = ratings.merge(links_small[['movieId', 'tmdbId']], on='movieId', how='left')
ratings_mapped = ratings_mapped.dropna(subset=['tmdbId'])
ratings_mapped['tmdbId'] = ratings_mapped['tmdbId'].astype('int64')

# Lọc phim có >= 5 rating
movie_rating_counts = ratings_mapped.groupby('tmdbId').size()
valid_movies = movie_rating_counts[movie_rating_counts >= 5].index
ratings_filtered = ratings_mapped[ratings_mapped['tmdbId'].isin(valid_movies)]

print("Ratings trước lọc:", ratings_mapped.shape[0])
print("Ratings sau lọc:", ratings_filtered.shape[0])
print("Số phim sau lọc:", ratings_filtered['tmdbId'].nunique())
print("Số user sau lọc:", ratings_filtered['userId'].nunique())

# Xây ma trận User-Item
user_item_matrix = ratings_filtered.pivot_table(index='userId', columns='tmdbId', values='rating', fill_value=0)
print("Shape ma trận User-Item:", user_item_matrix.shape)

Ratings trước lọc: 99933
Ratings sau lọc: 90015
Số phim sau lọc: 3493
Số user sau lọc: 671
Shape ma trận User-Item: (671, 3493)


In [8]:
import pickle

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models_artifacts', exist_ok=True)

# Lưu movies kèm soup + weighted_rating (dùng lại cho modeling, không cần tính lại)
movies[['id', 'title', 'soup', 'weighted_rating', 'vote_average', 'vote_count',
        'popularity', 'release_year', 'poster_path']].to_csv(
    '../data/processed/movies_features.csv', index=False)

# Lưu vectorizer + ma trận TF-IDF/Count
with open('../models_artifacts/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vec, f)
with open('../models_artifacts/tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix, f)
with open('../models_artifacts/count_vectorizer.pkl', 'wb') as f:
    pickle.dump(count_vec, f)
with open('../models_artifacts/count_matrix.pkl', 'wb') as f:
    pickle.dump(count_matrix, f)

# Lưu ma trận User-Item
with open('../models_artifacts/user_item_matrix.pkl', 'wb') as f:
    pickle.dump(user_item_matrix, f)

print("Đã lưu toàn bộ artifact cho bước Modeling")

Đã lưu toàn bộ artifact cho bước Modeling
